# Fine-tune VietOCR seq2seq trên VinText — Google Colab
Huấn luyện **đọc chữ**, không huấn luyện detector trường CCCD hoặc face matching.
Bật Runtime → Change runtime type → T4 GPU. Chạy SMOKE trước, sau đó đổi False và chạy lại từ cấu hình.
VinText là ảnh cảnh thực; cần đánh giá CCCD riêng, chưa bảo đảm cải thiện. Dataset gốc chỉ dùng nghiên cứu/giáo dục,
không phân phối lại dữ liệu/crop. Nguồn: https://github.com/VinAIResearch/dict-guided.
Notebook chưa được chạy trên GPU Colab trong phiên tạo; nếu cấu trúc Kaggle khác TXT gốc, dừng để kiểm tra.


In [ ]:
# Chạy trên runtime Colab GPU SẠCH. Nếu từng chạy cell numpy<2:
# Runtime -> Disconnect and delete runtime, kết nối GPU lại rồi chạy notebook này.
import json
import subprocess
import sys
from pathlib import Path

assert "google.colab" in sys.modules or Path("/content").is_dir(), "Notebook dành cho Google Colab"
probe = """import sys,json,numpy,torch,torchvision,cv2
from importlib.metadata import version
assert sys.version_info < (3,13) or int(numpy.__version__.split('.')[0]) >= 2, 'Python 3.13 cần NumPy tương thích; tạo runtime sạch'
assert torch.cuda.is_available(), 'Bật GPU T4 trong Runtime > Change runtime type'
x=numpy.zeros((8,8,3),dtype=numpy.uint8)
assert cv2.cvtColor(x,cv2.COLOR_RGB2GRAY).shape==(8,8)
t=torch.from_numpy(numpy.ones((2,2),dtype=numpy.float32)).cuda()
assert t.sum().item()==4
torchvision.ops.nms(torch.tensor([[0.,0.,2.,2.]],device='cuda'),torch.tensor([1.],device='cuda'),0.5)
print(json.dumps({'python':sys.version,'numpy':numpy.__version__,'torch':torch.__version__,'torchvision':torchvision.__version__,'opencv':cv2.__version__,'gpu':torch.cuda.get_device_name(0)}))
"""
check = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
print(check.stdout)
if check.returncode:
    print(check.stderr)
    raise RuntimeError(
        "Môi trường GPU nền chưa hợp lệ. Nếu đã hạ numpy: Disconnect and delete runtime; không chạy pip sửa chồng tiếp."
    )
# Khóa các gói nhị phân ở phiên bản Colab đang có; pip không được tự thay chúng.
from contextlib import suppress
from importlib.metadata import PackageNotFoundError, version

protected = [
    "numpy",
    "torch",
    "torchvision",
    "Pillow",
    "opencv-python",
    "opencv-contrib-python",
    "opencv-python-headless",
]
pins = []
for name in protected:
    with suppress(PackageNotFoundError):
        pins.append(name + "==" + version(name))
Path("/content/colab_native_constraints.txt").write_text("\n".join(pins) + "\n")
COLAB_ENV = json.loads(check.stdout.strip().splitlines()[-1])

In [ ]:
# Không cài/hạ numpy, OpenCV, torch hoặc torchvision; không kéo dependency training cũ của VietOCR.
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--only-binary=:all:",
        "-c",
        "/content/colab_native_constraints.txt",
        "einops",
        "gdown",
        "PyYAML",
        "requests",
        "kagglehub",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "--only-binary=:all:",
        "vietocr==0.3.13",
    ],
    check=True,
)
# Smoke test đúng các module notebook dùng, không import Trainer/imgaug/LMDB cũ.
import cv2
import numpy
import torch
import torchvision
from PIL import Image
from vietocr.model.vocab import Vocab
from vietocr.tool.translate import build_model, process_image

assert process_image(Image.new("RGB", (100, 32)), 32, 32, 512).shape == (3, 32, 100)
assert Vocab("aAêÊ ").decode(Vocab("aAêÊ ").encode("Ê a")) == "Ê a"
assert torch.cuda.is_available(), "Runtime phải có GPU"
print("Colab native stack + VietOCR imports OK:", numpy.__version__, torch.__version__)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
from pathlib import Path

SMOKE = True
RUN_NAME = "vintext_seq2seq_v1"  # đổi khi thay cấu hình; giữ để resume cùng cấu hình
BASE = Path("/content/drive/MyDrive/vericlue_vintext")
RUN = BASE / "runs" / (RUN_NAME + ("_smoke" if SMOKE else "_full"))
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-5
BASE.mkdir(parents=True, exist_ok=True)
print("Run:", RUN)

In [ ]:
import kagglehub

RAW = Path(kagglehub.dataset_download("hungkhoi/vietnamese-scene-text-spotting-dataset-vintext"))
print("Dataset:", RAW)
print("\n".join(str(p.relative_to(RAW)) for p in list(RAW.rglob("*"))[:50]))
# Nếu yêu cầu đăng nhập: dùng kagglehub.login(), không ghi token vào notebook chia sẻ.
# Nếu chỉ có archive, giải nén vào /content/vintext_raw rồi đặt RAW tới thư mục đó.

In [ ]:
from pathlib import Path

Path("/content/vintext_recognition.py").write_text(
    '"""Prepare original VinText quadrilateral TXT annotations for recognition only."""\n\nimport hashlib\nimport io\nimport json\nimport re\nimport time\nimport unicodedata\nfrom collections import Counter\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nfrom PIL import Image\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef annotation(line):\n    parts = line.strip("\\ufeff\\r\\n").split(",", 8)\n    if len(parts) != 9:\n        raise ValueError("Expected x1,y1,x2,y2,x3,y3,x4,y4,transcript")\n    points = np.asarray([float(v) for v in parts[:8]], dtype=np.float32).reshape(4, 2)\n    if not np.isfinite(points).all():\n        raise ValueError("Nonfinite polygon")\n    text = unicodedata.normalize("NFC", parts[8].strip())\n    return points, text\n\n\ndef rectify(image, points):\n    # Original annotation order is TL, TR, BR, BL; preserve reading direction.\n    h, w = image.shape[:2]\n    if (points < 0).any() or (points[:, 0] > w - 1).any() or (points[:, 1] > h - 1).any():\n        raise ValueError("Polygon outside image")\n    if not cv2.isContourConvex(points) or abs(cv2.contourArea(points)) < 4:\n        raise ValueError("Degenerate/nonconvex polygon")\n    width = round(max(np.linalg.norm(points[1] - points[0]), np.linalg.norm(points[2] - points[3])))\n    height = round(\n        max(np.linalg.norm(points[3] - points[0]), np.linalg.norm(points[2] - points[1]))\n    )\n    if min(width, height) < 2 or max(width, height) > 8192:\n        raise ValueError("Invalid crop size")\n    target = np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])\n    return cv2.warpPerspective(image, cv2.getPerspectiveTransform(points, target), (width, height))\n\n\ndef discover(root):\n    """Match original VinText im0001.jpg to gt_1.txt, without guessing splits."""\n    root = Path(root)\n    aliases = {\n        "train_images": "train",\n        "training": "train",\n        "test_image": "test",\n        "test_images": "test",\n        "test": "test",\n        "unseen_test_images": "unseen_test",\n        "unseen_test": "unseen_test",\n        # Kaggle names this partition val_image; do not claim it is original unseen_test.\n        "val_image": "dataset_val",\n    }\n    images, ignored_crops, other_image_dirs = {}, 0, Counter()\n    for p in sorted(root.rglob("*")):\n        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:\n            continue\n        # Kaggle bundles recognition crops beside the original scene images.\n        # A crop im0001_1 is not a scene im0001 and has no independent polygon TXT.\n        if re.fullmatch(r"im[0-9]+_[0-9]+", p.stem):\n            ignored_crops += 1\n            continue\n        splits = {aliases[x] for x in p.relative_to(root).parts[:-1] if x in aliases}\n        if len(splits) != 1:\n            other_image_dirs[str(p.parent.relative_to(root))] += 1\n            continue\n        if p.stem in images:\n            raise ValueError(f"Ambiguous image stem: {p.stem}")\n        images[p.stem] = (p, splits.pop())\n    txt_by_stem = {}\n    for p in sorted(root.rglob("*")):\n        if p.is_file() and p.suffix.lower() == ".txt":\n            txt_by_stem.setdefault(p.stem, []).append(p)\n    labels, owners = {}, {}\n    for stem in images:\n        candidates = {stem, "gt_" + stem}\n        match = re.fullmatch(r"im([0-9]+)", stem)\n        if match:\n            candidates.add("gt_" + str(int(match.group(1))))\n        found = {p for key in candidates for p in txt_by_stem.get(key, [])}\n        if len(found) > 1:\n            raise ValueError(f"Ambiguous annotation for {stem}: {sorted(map(str, found))}")\n        if found:\n            label = found.pop()\n            if label in owners:\n                raise ValueError(f"Annotation reused by {owners[label]} and {stem}: {label}")\n            owners[label] = stem\n            labels[stem] = label\n    split_counts = dict(Counter(s for _, s in images.values()))\n    diagnostics = {\n        "root": str(root),\n        "recognized_images": len(images),\n        "matched_labels": len(labels),\n        "official_split_counts": split_counts,\n        "ignored_recognition_crops": ignored_crops,\n        "unrecognized_image_directories": dict(other_image_dirs),\n    }\n    if not images or set(labels) != set(images):\n        raise ValueError(\n            "VinText layout or labels incomplete: "\n            + json.dumps(\n                {\n                    **diagnostics,\n                    "missing_label_examples": sorted(set(images) - set(labels))[:10],\n                    "file_examples": [\n                        str(p.relative_to(root)) for p in sorted(root.rglob("*")) if p.is_file()\n                    ][:30],\n                    "txt_stem_examples": sorted(txt_by_stem)[:10],\n                },\n                ensure_ascii=False,\n            )\n        )\n    available = set(split_counts)\n    if not {"train", "test"}.issubset(available) or not available.intersection(\n        {"unseen_test", "dataset_val"}\n    ):\n        raise ValueError("Missing official split: " + json.dumps(diagnostics, ensure_ascii=False))\n    print("VinText source discovery:", json.dumps(diagnostics, ensure_ascii=False))\n    return images, labels\n\n\ndef prepare(root, out, vocab, seed=2026):\n    root, out = Path(root), Path(out)\n    if out.exists():\n        raise ValueError("Choose new preparation folder; existing output is never overwritten")\n    images, labels = discover(root)\n    out.mkdir(parents=True)\n    (out / "crops").mkdir()\n    rows, rejected, provenance, pixel_splits = [], [], [], {}\n    started = last_report = time.monotonic()\n    print(\n        f"Preparing {len(images)} source images -> {out}; CPU crop/PNG and disk I/O, not GPU training",\n        flush=True,\n    )\n    for image_index, (stem, (path, official)) in enumerate(images.items(), 1):\n        with Image.open(path) as source:\n            rgb = np.array(source.convert("RGB"))\n        pixel = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()\n        if pixel in pixel_splits and pixel_splits[pixel] != official:\n            raise ValueError("Identical source image crosses official splits")\n        pixel_splits[pixel] = official\n        # Hold out whole source-image pixel groups from training, never individual crops.\n        split = (\n            "valid"\n            if official == "train"\n            and int(hashlib.sha256(f"{seed}:{pixel}".encode()).hexdigest()[:8], 16) % 10 == 0\n            else official\n        )\n        provenance.append(\n            {\n                "image": str(path.relative_to(root)),\n                "image_sha256": digest(path),\n                "annotation": str(labels[stem].relative_to(root)),\n                "annotation_sha256": digest(labels[stem]),\n                "official_split": official,\n                "split": split,\n                "pixel_sha256": pixel,\n            }\n        )\n        for index, line in enumerate(labels[stem].read_text(encoding="utf-8-sig").splitlines()):\n            now = time.monotonic()\n            if now - last_report >= 10:\n                print(\n                    f"Preparing image {image_index}/{len(images)} ({path.name}), "\n                    f"line {index + 1}; crops={len(rows)}, rejected={len(rejected)}, "\n                    f"elapsed={now - started:.0f}s",\n                    flush=True,\n                )\n                last_report = now\n            if not line.strip():\n                continue\n            try:\n                points, text = annotation(line)\n                if text in {"", "###", "***"}:\n                    raise ValueError("Ignored/unreadable transcript")\n                if (\n                    any(c not in vocab for c in text)\n                    or len(text) > 120\n                    or any(c in text for c in "\\t\\r\\n")\n                ):\n                    raise ValueError("Unsupported vocabulary or transcript length")\n                crop = rectify(rgb, points)\n            except ValueError as exc:\n                rejected.append(\n                    {"image": str(path.relative_to(root)), "line": index + 1, "reason": str(exc)}\n                )\n                continue\n            uid = hashlib.sha256(f"{pixel}:{index}:{text}".encode()).hexdigest()[:24]\n            target = out / "crops" / f"{uid}.png"\n            # Hash encoded bytes in memory instead of reading each PNG back from Drive.\n            buffer = io.BytesIO()\n            Image.fromarray(crop).save(buffer, format="PNG")\n            encoded = buffer.getvalue()\n            target.write_bytes(encoded)\n            rows.append(\n                {\n                    "crop": f"crops/{uid}.png",\n                    "crop_sha256": hashlib.sha256(encoded).hexdigest(),\n                    "text": text,\n                    "split": split,\n                    "source_pixel_sha256": pixel,\n                }\n            )\n    counts = Counter(r["split"] for r in rows)\n    print(f"Crops complete: {dict(counts)}; writing manifest/audit...", flush=True)\n    expected = {s for _, s in images.values()} | {"valid"}\n    if set(counts) != expected:\n        raise ValueError("Empty split after filtering; inspect annotations")\n    (out / "manifest.json").write_text(\n        json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    (out / "audit.json").write_text(\n        json.dumps(\n            {\n                "counts": dict(counts),\n                "rejected": rejected,\n                "sources": provenance,\n                "seed": seed,\n                "scope": "Recognition crops, original TXT format; valid held out from official train by source pixels; scene/near-duplicate leakage not audited",\n            },\n            ensure_ascii=False,\n            indent=2,\n        ),\n        encoding="utf-8",\n    )\n    return dict(counts)\n',
    encoding="utf-8",
)

In [ ]:
from pathlib import Path

Path("/content/train_vintext.py").write_text(
    '"""VietOCR recognition fine-tuning with resumable epoch checkpoints and held-out validation."""\n\nimport json\nimport random\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport yaml\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Dataset\nfrom vintext_recognition import digest\n\n\ndef distance(a, b):\n    prev = list(range(len(b) + 1))\n    for i, x in enumerate(a, 1):\n        cur = [i]\n        for j, y in enumerate(b, 1):\n            cur.append(min(cur[-1] + 1, prev[j] + 1, prev[j - 1] + (x != y)))\n        prev = cur\n    return prev[-1]\n\n\nclass Crops(Dataset):\n    def __init__(self, root, rows, vocab):\n        self.root, self.rows, self.vocab = Path(root), rows, vocab\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        from vietocr.tool.translate import process_image\n\n        row = self.rows[index]\n        path = (self.root / row["crop"]).resolve()\n        if not path.is_relative_to(self.root.resolve()):\n            raise ValueError("Crop outside dataset")\n        with Image.open(path) as source:\n            image = process_image(source, 32, 32, 512)\n        return torch.tensor(image, dtype=torch.float32), self.vocab.encode(row["text"])\n\n\ndef collate(items):\n    width = max(im.shape[2] for im, _ in items)\n    length = max(len(tokens) - 1 for _, tokens in items)\n    images = torch.ones(len(items), 3, 32, width)\n    inputs = torch.zeros(length, len(items), dtype=torch.long)\n    targets = torch.zeros(len(items), length, dtype=torch.long)\n    for i, (im, tokens) in enumerate(items):\n        images[i, :, :, : im.shape[2]] = im\n        inputs[: len(tokens) - 1, i] = torch.tensor(tokens[:-1])\n        targets[i, : len(tokens) - 1] = torch.tensor(tokens[1:])\n    return images, inputs, targets\n\n\ndef evaluate(model, vocab, dataset, device):\n    from vietocr.tool.translate import translate\n\n    errors = chars = exact = 0\n    model.eval()\n    with torch.inference_mode():\n        for i, row in enumerate(dataset.rows):\n            image, _ = dataset[i]\n            ids, _ = translate(image.unsqueeze(0).to(device), model, max_seq_length=128)\n            prediction = vocab.decode(ids[0].tolist())\n            expected = row["text"]\n            errors += distance(prediction, expected)\n            chars += len(expected)\n            exact += prediction == expected\n    if not len(dataset) or not chars:\n        raise ValueError("Empty evaluation")\n    return {"samples": len(dataset), "cer": errors / chars, "exact_match": exact / len(dataset)}\n\n\ndef save(path, value):\n    temporary = path.with_suffix(".tmp")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef train(dataset, model_dir, out, smoke=True, epochs=10, batch_size=16, lr=1e-5):\n    from vietocr.tool.translate import build_model\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable Colab T4 GPU before training")\n    if epochs < 1 or batch_size < 1 or not 0 < lr < 1:\n        raise ValueError("Invalid training settings")\n    dataset, model_dir, out = map(Path, (dataset, model_dir, out))\n    rows = json.loads((dataset / "manifest.json").read_text(encoding="utf-8"))\n    selected = {s: [r for r in rows if r["split"] == s] for s in ("train", "valid")}\n    if smoke:\n        selected = {s: rs[: 32 if s == "train" else 16] for s, rs in selected.items()}\n        epochs = 1\n    if any(not rs for rs in selected.values()):\n        raise ValueError("Empty train/valid split")\n    if {r["source_pixel_sha256"] for r in selected["train"]} & {\n        r["source_pixel_sha256"] for r in selected["valid"]\n    }:\n        raise ValueError("Train/valid source leakage")\n    for rs in selected.values():\n        for row in rs:\n            path = (dataset / row["crop"]).resolve()\n            if not path.is_relative_to(dataset.resolve()) or digest(path) != row["crop_sha256"]:\n                raise ValueError("Crop changed or unsafe path")\n    config = yaml.safe_load((model_dir / "local.yml").read_text(encoding="utf-8"))\n    config["device"] = "cuda:0"\n    config["cnn"]["pretrained"] = False\n    config["predictor"]["beamsearch"] = False\n    initial = model_dir / "vgg_seq2seq.pth"\n    signature = {\n        "manifest": digest(dataset / "manifest.json"),\n        "initial_weights": digest(initial),\n        "config": digest(model_dir / "local.yml"),\n        "smoke": smoke,\n        "epochs": epochs,\n        "batch_size": batch_size,\n        "lr": lr,\n        "script": digest(__file__),\n        "versions": {n: version(n) for n in ("torch", "torchvision", "vietocr", "numpy", "Pillow")},\n    }\n    out.mkdir(parents=True, exist_ok=True)\n    last = out / "last.pt"\n    if not last.exists() and any(out.iterdir()):\n        raise ValueError("Run folder contains files but no checkpoint; choose new run")\n    checkpoint = torch.load(last, map_location="cpu", weights_only=False) if last.exists() else None\n    if checkpoint and checkpoint["signature"] != signature:\n        raise ValueError(\n            "Checkpoint incompatible: choose new RUN_NAME; never overwrite smoke with full training"\n        )\n    torch.manual_seed(2026)\n    random.seed(2026)\n    np.random.seed(2026)\n    model, vocab = build_model(config)\n    model.load_state_dict(torch.load(initial, map_location="cpu", weights_only=True), strict=True)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n    data = {s: Crops(dataset, rs, vocab) for s, rs in selected.items()}\n    history = []\n    best = float("inf")\n    start = 0\n    if checkpoint:\n        model.load_state_dict(checkpoint["model"])\n        optimizer.load_state_dict(checkpoint["optimizer"])\n        start = checkpoint["epoch"] + 1\n        best = checkpoint["best"]\n        history = checkpoint["history"]\n        torch.set_rng_state(checkpoint["rng"])\n        torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])\n    else:\n        baseline = evaluate(model, vocab, data["valid"], "cuda:0")\n        history.append({"epoch": -1, "validation": baseline})\n        best = baseline["cer"]\n        save(out / "best.pt", model.state_dict())\n        config["weights"] = "best.pt"\n        config["device"] = "cpu"\n        (out / "local.yml").write_text(yaml.safe_dump(config, allow_unicode=True), encoding="utf-8")\n        print("Pretrained validation:", baseline, flush=True)\n    for epoch in range(start, epochs):\n        generator = torch.Generator().manual_seed(2026 + epoch)\n        loader = DataLoader(\n            data["train"],\n            batch_size=batch_size,\n            shuffle=True,\n            collate_fn=collate,\n            num_workers=0,\n            generator=generator,\n        )\n        losses = []\n        for images, inputs, targets in loader:\n            model.train()\n            optimizer.zero_grad(set_to_none=True)\n            images, inputs, targets = [x.to("cuda:0") for x in (images, inputs, targets)]\n            outputs = model(images, inputs, tgt_key_padding_mask=targets.eq(0))\n            loss = torch.nn.functional.cross_entropy(\n                outputs.reshape(-1, outputs.shape[-1]), targets.reshape(-1), ignore_index=0\n            )\n            if not torch.isfinite(loss):\n                raise RuntimeError("Nonfinite loss; stopped before saving checkpoint")\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer.step()\n            losses.append(loss.item())\n        metrics = evaluate(model, vocab, data["valid"], "cuda:0")\n        if metrics["cer"] < best:\n            best = metrics["cer"]\n            save(out / "best.pt", model.state_dict())\n        history.append({"epoch": epoch, "loss": sum(losses) / len(losses), "validation": metrics})\n        save(\n            last,\n            {\n                "signature": signature,\n                "epoch": epoch,\n                "model": model.state_dict(),\n                "optimizer": optimizer.state_dict(),\n                "best": best,\n                "history": history,\n                "rng": torch.get_rng_state(),\n                "cuda_rng": torch.cuda.get_rng_state_all(),\n            },\n        )\n        (out / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n        print(history[-1], flush=True)\n    return history\n',
    encoding="utf-8",
)

In [ ]:
import hashlib

import requests
import yaml

MODELS = BASE / "pretrained"
MODELS.mkdir(exist_ok=True)
cfg = {}
for name in ["base.yml", "vgg-seq2seq.yml"]:
    response = requests.get("https://vocr.vn/data/vietocr/config/" + name, timeout=60)
    response.raise_for_status()
    cfg.update(yaml.safe_load(response.text))
weight = MODELS / "vgg_seq2seq.pth"
if not weight.exists():
    temporary = weight.with_suffix(".part")
    with requests.get(cfg["weights"], stream=True, timeout=120) as response:
        response.raise_for_status()
        with temporary.open("wb") as f:
            for block in response.iter_content(1024 * 1024):
                f.write(block)
    temporary.replace(weight)
assert (
    hashlib.sha256(weight.read_bytes()).hexdigest()
    == "0921503a41375a0584268e23ef3d414ea478a8fe8777865c7745d38f2d0bc5db"
), "Pretrained weight mismatch"
cfg["cnn"]["pretrained"] = False
cfg["weights"] = str(weight)
cfg["device"] = "cpu"
(MODELS / "local.yml").write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding="utf-8")

In [ ]:
import importlib

import vintext_recognition

importlib.reload(vintext_recognition)
PREP = BASE / "prepared_v1"
if not PREP.exists():
    print(vintext_recognition.prepare(RAW, PREP, cfg["vocab"]))
import json

assert (PREP / "audit.json").exists(), "Chuẩn bị chưa hoàn tất: dùng thư mục PREP mới để chạy lại"
audit = json.loads((PREP / "audit.json").read_text())
print("Crops:", audit["counts"], "Rejected:", len(audit["rejected"]))
print(
    "Kiểm tra audit.json và crop trước khi train thật; test/dataset_val/unseen_test không dùng để chọn model."
)
from IPython.display import display
from PIL import Image

rows = json.loads((PREP / "manifest.json").read_text())
for row in [r for r in rows if r["split"] == "train"][:8]:
    print(row["text"])
    display(Image.open(PREP / row["crop"]))

In [ ]:
import train_vintext

importlib.reload(train_vintext)
history = train_vintext.train(
    PREP, MODELS, RUN, smoke=SMOKE, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE
)
print("Best checkpoint:", RUN / "best.pt")
print(
    "Nếu validation không tốt hơn pretrained, best.pt vẫn là pretrained. Không tự ghi đè model dự án."
)

In [ ]:
# Chỉ chạy sau khi đã chốt cấu hình từ validation; không dùng test để chỉnh hyperparameter.
EVALUATE_FINAL_TEST = False
if EVALUATE_FINAL_TEST:
    assert not SMOKE, "Không báo test cho smoke run"
    from vietocr.tool.translate import build_model

    config = yaml.safe_load((RUN / "local.yml").read_text())
    config["device"] = "cuda:0"
    model, vocab = build_model(config)
    results = {}
    for label, weights in [
        ("pretrained", MODELS / "vgg_seq2seq.pth"),
        ("selected", RUN / "best.pt"),
    ]:
        model.load_state_dict(
            torch.load(weights, map_location="cpu", weights_only=True), strict=True
        )
        results[label] = {}
        for split in sorted({r["split"] for r in rows} - {"train", "valid"}):
            data = train_vintext.Crops(PREP, [r for r in rows if r["split"] == split], vocab)
            results[label][split] = train_vintext.evaluate(model, vocab, data, "cuda:0")
    target = RUN / "final_test.json"
    with target.open("x", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(results)

In [ ]:
# ZIP chỉ gồm model/config/kết quả; không đóng gói dữ liệu VinText.
import zipfile

archive = RUN / "vietocr_vintext_export.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(RUN / "best.pt", "vgg_seq2seq.pth")
    export_cfg = yaml.safe_load((RUN / "local.yml").read_text())
    export_cfg["weights"] = "vgg_seq2seq.pth"
    z.writestr("local.yml", yaml.safe_dump(export_cfg, allow_unicode=True))
    for name in ["history.json", "final_test.json"]:
        if (RUN / name).exists():
            z.write(RUN / name, name)
    z.writestr(
        "scope.txt",
        f"SMOKE={SMOKE}; VinText scene text recognition; CCCD performance not established. No dataset included.",
    )
from google.colab import files

files.download(str(archive))